# STUFF LEARNED AFTER DOING THE LAB

# WORKFLOW
> #### Initial inspection
- What is the target? Name of the target column?
- Data type of the target: continuous = regression OR categorical = classification?
- Is it supervised or unsupervised learning?
- Inspect number of rows and columns in train and test set. Do they have the same features?
**.describe()**  
**.info()**  

---

> #### FIRST LAYER: drop columns / rows based on Nan values
- Inspect missing values PER COLUMN:
    - compute missing value % per feature:
        - missing_rate < 0.05 → keep, simple impute
        - 0.05 ≤ missing_rate ≤ 0.3 → keep, better impute --> (SimpleImputer) / KNNImputer --> **FOR NUMERIC COLUMNS**
        - missing_rate > 0.5 → usually drop unless clearly important
```python
DF_train.isna().sum()/DF_train.shape[0]*100**
```

- Inspect missing values PER ROW:
    - see how many Nan values there are per row
    - based on that fix a threshold of Nan values that a row can have --> drop rows with Nan count > threshold

```python
row_na_counts = train_data.isna().sum(axis=1)
row_na_counts_sorted = row_na_counts.sort_values(ascending=False)
train_data = train_data[train_data.isna().sum(axis=1) < 20 ].reset_index(drop=True)
```

---

> #### IDENTIFY 3 DATA TYPES
> - *Inspect all columns data type and divide the columns in:*  
>     - **numeric features**  
>     - **ordinal features** : ord_7_val_3, ord_7_val_4, ord_7_val_5  
>     - **categorical features** : cat_0_val_1, cat_0_val_2, cat_0_val_3  

---

> #### Visualize distribution PER data type
- visualize continuous features distribution: plot --> continuous = histogram
```python
plt.figure(figsize=(20, 15))

for i, col in enumerate(continuous_features):
    subplot_index = i + 1
    plt.subplot(6, 5, subplot_index)            # builds a matrix 6,5 and specifies the index position (subplot_index) of the cell of the matrix to draw in
    sns.histplot(train_data[col], kde=True)
    plt.title(col)

plt.tight_layout()
plt.show()
```

- visualize ordinal features distribution: plot --> ordinal = countplot (counts frequencies)
```python
plt.figure(figsize=(20, 15))
for i, col in enumerate(ordinal_features):
    subplot_index = i + 1
    plt.subplot(5, 4, subplot_index)
    
    sns.countplot(x=train_data[col])
    
    # count categories
    # value_counts = train_data[col].value_counts()
    # bar plot
    # plt.bar(value_counts.index.astype(str), value_counts.values)
    
    plt.xticks(rotation=45)
    plt.title(col)

    plt.xticks(rotation=45)
    plt.title(col)

plt.tight_layout()
plt.show()
```

- visualize categorical features distribution: plot --> ordinal = countplot (counts frequencies)
```python
plt.figure(figsize=(20, 15))
for i, col in enumerate(categorical_features):
    subplot_index = i + 1
    plt.subplot(5, 4, subplot_index)
    sns.countplot(x=train_data[col])
    plt.xticks(rotation=45)
    plt.title(col)

plt.tight_layout()
plt.show()
```

---

> #### Identify anomalies in plots --> SECOND LAYER: drop no info gain columns/fetaures
- look at these graphs and look for 'anomalies':

    - for the continuous featires histograms:
        - The histogram has one very tall bar and almost nothing elsewhere.
        - If the histogram has one huge bar and the rest is almost empty --> the feature is basically constant --> probably useless.
        - If one feature has a few extreme huge values → maybe apply log(x+1) to reduce their impact --> Log / Box-Cox transform.

    - for the ordinal features bar charts:
        - look for plots with just one bar --> that column has only one value --> no info gain --> **DROP IT**

    - for the categorical features bar charts:
        - look for plots with just one bar --> that column has only one value --> no info gain --> **DROP IT**
        - look for features with **too many categories** (tons of tiny bars) --> high cardinality --> think about:
            - grouping rare categories into “Other”
            - or even dropping if clearly useless

**That's the end of data inspection and selection**

---

> #### [OPTIONAL BUT VERY FANCY] Correlation of continuous features with target (for feature selection / poly)
Compute the linear correlation between continuous features and the target, why?
Instead of aplying PolynomialFeatures on ALL the features, which would explode the number of rows to thousands of rows, we select only some special features on which we perform the PolynomialFeatures:

Work only on the **training data** (never on test):
```python
# compute correlation between each continuous feature and target
cont_corr = train_data[continuous_features].corrwith(train_data['target'])

# take absolute value: even strong negative correlation is still “strong”
cont_corr_abs = cont_corr.abs()

# sort by |correlation| descending
cont_corr_sorted = cont_corr_abs.sort_values(ascending=False)

# pick top k continuous features (e.g. top 5)
top_continuous = cont_corr_sorted.head(5).index.tolist()

print("Top correlated continuous features:", top_continuous)
print("Top correlations:", cont_corr_sorted.head(5).values)
```

> top_continuous = the numeric features most linearly related to the target.  

You can use this to:  
- apply PolynomialFeatures only to these,
- keep the others as “simple scaled numeric” (no poly).

```python
# numeric features that will get PolynomialFeatures (top_continuous)
num_poly_pipeline = make_pipeline(
    SimpleImputer(strategy='median'),
    StandardScaler(),
    PolynomialFeatures(degree=2, include_bias=False)
)

# numeric features that will just be scaled (other_continuous)
num_simple_pipeline = make_pipeline(
    SimpleImputer(strategy='median'),
    StandardScaler()
)

# ORDINAL pipeline (same idea as yours, but with imputer)
ord_preprocessing_pipeline = make_pipeline(
    SimpleImputer(strategy='most_frequent'),
    OrdinalEncoder(
        handle_unknown='use_encoded_value',
        unknown_value=-1,
        encoded_missing_value=-1,
        dtype=int
    )
)

# CATEGORICAL pipeline
cat_preprocessing_pipeline = make_pipeline(
    SimpleImputer(strategy='most_frequent'),
    OneHotEncoder(handle_unknown='ignore')
)

# ColumnTransformer tying everything together
preprocessing = ColumnTransformer(
    transformers=[
        # numeric with poly
        ('numeric_poly',   num_poly_pipeline,   top_continuous),
        # numeric without poly
        ('numeric_simple', num_simple_pipeline, other_continuous),
        # ordinal
        ('ordinal_data',   ord_preprocessing_pipeline, ordinal_columns),
        # categorical
        ('categorical_data', cat_preprocessing_pipeline, categorical_columns),
    ]
)
```

---

> #### Data preprocessing: building pipelines + pipeline tools
We'll have 3 different pipelines, one for each data type.  
This is what we did in THIS LAB:

- **Numerical** data pipeline:
    - We dropped the useless columns and rows, but we still have some Nan values --> imputer: **SimpleImputer()** OR more fancy **KNNImputer()**
    - If the numerical data IS NOT standardized (mean = 0 std  = 1) --> **StandardScaler()**

- **Ordinal** data pipeline:
    - We dropped the useless columns and rows, but we still have some Nan values --> imputer: **SimpleImputer(strategy = 'most_frequent')**
    - IF the ordinal data has a meaningful order, like: low, medium, high --> **OrdinalEncoder** (strings --> numbers)

- **Categorical** data pipeline:
    - We dropped the useless columns and rows, but we still have some Nan values --> imputer: **SimpleImputer(strategy = 'most_frequent')**
    - We need to convert strings into numbers --> **OneHotEncoder()** OR use the Pandas method **.get_dummies**, which does the same exact thing

This is what we do for this lab, BUT we could have used many other tools inside the pipelines, like:  
- MinMaxScaler
    - Scales numbers to a fixed range, usually `[0, 1]`.
    - Good if the model is sensitive to the scale (e.g. KNN, neural nets) and you want “everything between 0 and 1” instead of mean 0, std 1.

- RobustScaler
    - Similar idea to `StandardScaler`, but uses **median** and **IQR** instead of mean and std.
    - More robust when you have **strong outliers** (it doesn’t get destroyed by a few crazy values).

- **PolynomialFeatures**
    - Creates new features from the old ones:
        - squares (x²), cubes (x³),
        - interactions (x₁·x₂, x₁·x₂², …)
    - Used to give linear models some **non-linear power**.
    - Warning: number of features can explode.

- SelectKBest (feature selection)
    - Keeps only the “top K” features according to some score (e.g. correlation with the target).
    - Idea: remove weak / noisy features before training.
    - In sklearn: `SelectKBest(f_regression, k=...)` for regression.

- PCA (Principal Component Analysis)
    - Compresses many correlated features into a smaller number of new features (principal components).
    - These new features still keep most of the variance (information).
    - Good when you have **lots of numeric features** and want to reduce dimensionality.

---

> #### In general how do we decide what to do in the preprocessig?
1.	First: decide 1–2 model families you want to try:  
- e.g. “linear-ish + trees” (Ridge + RandomForest).

2.	Then: design preprocessing compatible with those:  
- for Ridge: impute + scale + (maybe) polynomial on numeric, OHE on cats.
- for RF: impute + encode cats, no scaling, no poly.

> You do not want 100 different preprocessing setups, each bespoke to a single model. You’ll drown.

In practice you do:
- 1 preprocessing pipeline for “scaled, linear-ish models”
- 1 preprocessing pipeline for “tree-ish models”

> 1.	Baseline preprocessing (simple, sane) --> **STARTING POINT**:  
- numeric: SimpleImputer(median) + StandardScaler (don't use it for trees)
- ordinal: SimpleImputer + OrdinalEncoder
- categorical: SimpleImputer + OneHotEncoder

> 2.	Choose one reference model (e.g. Ridge for linear stack, RF for tree stack).  

> 3.	Use GridSearchCV/RandomizedSearchCV on a small set of preprocessing hyperparams:  
- e.g. imputer__strategy: mean vs median
- polynomialfeatures__degree: 1 vs 2 vs 3
- maybe presence/absence of certain transforms (with optional steps).

> 4.	Look at CV scores:  
- if variant A ≈ variant B → pick the simpler one.
- if variant A clearly better → pick that.

> 5.	Freeze that preprocessing pipeline for that model family.  
> **Then move to model hyperparameters**

---

> #### Tie pipelines
To connect all these preprocessing pipelines use **ColumnTransformer**:  
```python
# pipeline for numeric data
num_preprocessing_pipeline = make_pipeline(
    StandardScaler()
)

# pipeline for ordinal data
ord_preprocessing_pipeline = make_pipeline(
    OrdinalEncoder(
        handle_unknown='use_encoded_value',
        unknown_value=-1,
        encoded_missing_value=-1,
        dtype=int)
)

# pipeline for categorical data
cat_preprocessing_pipeline = make_pipeline(
    OneHotEncoder(handle_unknown='ignore')
)

# connect the two pipelines
preprocessing = ColumnTransformer(
    transformers = [
        ('numeric_data', num_preprocessing_pipeline, numeric_columns),
        ('categorical_data', cat_preprocessing_pipeline, categorical_columns),
        ('ordinal_data', ord_preprocessing_pipeline, ordinal_columns)
    ]
)
```

---

#### Check the 'goodness' of the preprocessing pipeline trying it on models
First thing to do is see if the preprocessing pipeline is broken or not:
- check for dimensions, to make sure it didn't explode and has 1000000 columns
- check if there are any Nan --> if so it's broken

Nice, if it's not broken, now the idea is trying the preprocesing pipeline, the idea is trying it out on standard models, see how they perform, try to tune the preporcessing, adding / removing layers / tuning preporcessing hyperparameters with gridsearch.

Try feeding this preprocessing to some "standard models" that makes sense for that preprocessing:
- use **Linear / Ridge / Lasso / ElasticNet** if you're pipeline is like:
  - numeric → impute (SimpleImputer) + **scale** (StandardScaler); optional PolynomialFeatures
  - ordinal → impute + OrdinalEncoder → (optionally) scale together with numeric
  - categorical → impute + OneHotEncoder / get_dummies

- **KNN / SVR**:
  - numeric → impute + **strong scaling** (StandardScaler / MinMaxScaler)
  - ordinal → impute + OrdinalEncoder + scaling
  - categorical → impute + OneHotEncoder (but high dimensionality hurts KNN/SVR)

- **Tree-based (DecisionTree, RandomForest, GradientBoosting, XGB, etc.)**:
  - numeric → impute only, **no scaling, no PolynomialFeatures**
  - ordinal → impute + OrdinalEncoder (or integer codes)
  - categorical → impute + OneHotEncoder or native cat support (if the library has it)

- **Polynomial regression** (Linear/Ridge on PolynomialFeatures):
  - numeric → impute + scale + PolynomialFeatures (degree small, e.g. 2–3)
  - categorical → usually just impute + OneHotEncoder  
    → DON’T expand polys on all dummy columns blindly (feature explosion, overfitting).


> More in general:

| Model                      | Numeric features                                        | Ordinal features                                           | Categorical features                                              | Typical use / notes                                                                                     |
|----------------------------|---------------------------------------------------------|------------------------------------------------------------|-------------------------------------------------------------------|---------------------------------------------------------------------------------------------------------|
| **LinearRegression**       | Impute (SimpleImputer), **StandardScaler** important if features on very different scales. Optional **PolynomialFeatures** if you want non-linearity. | **OrdinalEncoder** (keep the order!), then scaled together with numeric if you want. | **OneHotEncoder** (or `get_dummies`), usually `drop='first'`.     | Baseline linear model. Use when relationship is (roughly) linear or you want something simple/interpretable. |
| **Ridge**                  | Same as LinearRegression: impute + **StandardScaler**, PolynomialFeatures if you want non-linear but still linear-in-parameters. | Same as above: OrdinalEncoder, then scale with numeric.    | OneHotEncoder / get_dummies.                                      | Like LinearRegression but with **L2 regularization**. Good when many correlated features. Very common with poly features. |
| **Lasso**                  | Impute + StandardScaler. Often **strongly regularized**, so scaling is mandatory. Poly features possible but can get nasty. | OrdinalEncoder + scaling.                                  | OneHotEncoder.                                                    | Useful when you want **feature selection** (sparse coefficients). Less stable with tons of correlated poly+OHE features. |
| **ElasticNet**             | Impute + StandardScaler. Yields mix of Ridge + Lasso behaviour. | OrdinalEncoder + scaling.                                  | OneHotEncoder.                                                    | Compromise between Ridge and Lasso. Good when you suspect many correlated but some irrelevant features. |
| **KNNRegressor**           | Impute, **StandardScaler or MinMaxScaler is crucial** (distance-based). No PolynomialFeatures unless you know what you’re doing → distances explode. | OrdinalEncoder, then scale like numeric.                   | OneHotEncoder; consider reducing very high-cardinality cats.      | Distance-based model. Needs scaling. Works well in low-dim / clean numeric spaces. Can be bad with many dummies. |
| **SVR (RBF kernel)**       | Impute, **StandardScaler or MinMaxScaler mandatory**. Often no PolynomialFeatures (kernel already adds non-linearity). | OrdinalEncoder + scaling.                                  | OneHotEncoder; keep number of features under control.             | Powerful in small-to-medium dimensional datasets. Very sensitive to scaling and hyperparameters.        |
| **DecisionTreeRegressor**  | Impute (SimpleImputer), **no scaling needed**. No PolynomialFeatures: tree already splits in non-linear ways. | OrdinalEncoder OR even treat them as numeric integers (but careful with arbitrary codes). | OneHotEncoder or sometimes even LabelEncoding (tree doesn’t care about metric distance, but OHE is safer). | Single tree: interpretable but often overfits. Good for understanding splits, not best predictive performance alone. |
| **RandomForestRegressor**  | Impute, **no scaling**, no PolynomialFeatures.         | OrdinalEncoder or treat as integers (still better to encode properly). | OneHotEncoder or leave as categories if using libs that support categorical splits. | Strong default for tabular data. Robust, handles non-linearities and interactions automatically. Scaling is typically useless. |
| **GradientBoosting / XGB / LightGBM** | Impute, no scaling, no PolynomialFeatures. | OrdinalEncoder / integer codes.                             | OneHotEncoder OR native categorical handling (depends on library). | Very strong on tabular data. Captures complex patterns. Preprocessing mostly = “just encode + impute”.  |
| **Polynomial Regression (Linear/Ridge on PolynomialFeatures)** | Impute + **StandardScaler strongly recommended**, **PolynomialFeatures**(degree 2–3 usually). | OrdinalEncoder, then polynomial expansion if you really think interactions with ordinals matter. | OneHotEncoder, but be careful: don’t usually apply PolynomialFeatures to the OHE part (feature explosion). | When you believe underlying relationship is smooth and continuous and want explicit polynomial terms. Easy to overfit. |
| **GaussianProcessRegressor** | Impute + StandardScaler. No need for PolynomialFeatures. | OrdinalEncoder + scaling.                                  | OneHotEncoder (high cardinality = pain).                          | Very flexible but scales terribly with many samples. Used on small datasets. Scaling is important.      |
| **HuberRegressor / RANSAC** | Impute + StandardScaler. Optional PolynomialFeatures. | OrdinalEncoder + scaling.                                  | OneHotEncoder.                                                    | Robust linear(-ish) models when you have **outliers** and still want something like a linear model.     |

---

#### Build a dictionary of models --> so the grid search on all the models to find the best one
Now the preprocessing is fixed and so should be the standard model we'll use --> pick the model that performs best with the preprocessing.

> NOW, we tune the model's hyperparameters

Build a dictionary of models that will be structured like this:
```python
models = {}
{
    'RandomForest' : {
        'model' : RandomForest model,
        'params' : {
            'n_estimators': [10, 20, 50],
            'max_depth': [10, 20],
            'min_samples_split': [2, 5]
            }
    }
    ,
    'KNN' : {
        'model' : KNeighborsRegressor,
        'params' : {
            'n_neighbors': [5, 10],
            'weights': ['uniform', 'distance'],
            'p': [1, 2]
            }
    }
    ....
}
```

#### Build final pipeline + perform GridSearch with fixed preprocessing to find the best models
We'll feed the Gridsearch the final pipeline which will contain the preprocessing + regression model and we'll give the GridSearch the hyperparameter grid used for tuning the regresssion model (not the preprocessing, as we already seacrhed the 'best' processing and fixed it before). 

```python
best_models = {}

for model_name, mp in models.items():
    print(f"Training {model_name}...")

    base_model = mp['model']
    param_grid = mp['params']

    # build the full pipeline: preprocessing + model
    pipe = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('model', base_model)
    ])

    grid = GridSearchCV(
        estimator=pipe,
        param_grid=param_grid,
        cv=3,
        scoring='neg_mean_squared_error',
        n_jobs=-1
    )

    grid.fit(X, y)      # X, y are still the **raw** train_data (with mixed types)

    best_estimator = grid.best_estimator_
    best_params = grid.best_params_
    best_score = grid.best_score_      # negative MSE

    best_models[model_name] = best_estimator

    avg_mse = -best_score
    print(f"{model_name} best params: {best_params}")
    print(f"{model_name} MSE: {avg_mse}\n")
```

---

#### Among the best models pick THE best model
If, for example, the RandomForestRegressor scores high pick that one.  
Evaluate the best model on the evaluation data.

```python
best_rf = best_models['RandomForest']

# Evaluate with a train/eval split if you want:
X_train, X_eval, y_train, y_eval = train_test_split(X, y, test_size=0.2, random_state=42)

best_rf.fit(X_train, y_train)
y_pred = best_rf.predict(X_eval)

mse_eval = mean_squared_error(y_eval, y_pred)
r2_eval = r2_score(y_eval, y_pred)
print("Eval MSE:", mse_eval)
print("Eval R2:", r2_eval)
```

> #### Finally predict the test data
```python
chosen_model = best_models['KNN']

test_predictions = chosen_model.predict(test_data)

submission = pd.DataFrame()
submission['index'] = np.arange(len(test_predictions))
submission['value'] = test_predictions

submission.to_csv('submission.csv', index=False)
```

---

# Other preprocessing stuff
#### 1. TF–IDF and TfidfVectorizer
What is TF–IDF?  
It’s a way to turn text into numeric vectors that a model can use.  

For each word t in a document d:  
1. TF (term frequency)  
How often the word appears in that document:  
$TF(t, d) = \frac{\text{count of } t \text{ in } d}{\text{total number of words in } d}$

2. IDF (inverse document frequency)  
How “rare” or “informative” the word is across the corpus:  
$IDF(t) = \log \frac{N}{df(t)}$

- N = total number of documents
- df(t) = number of documents containing t

3. TF–IDF  
$TFIDF(t, d) = TF(t, d) \cdot IDF(t)$

So:  
- Very common words across all documents (like “the”, “and”) get low weight.
- Words that are frequent in a specific document but rare in the corpus get high weight.

It’s basically a weighted “importance” of each word per document.  
It does not understand meaning, synonyms, etc. It’s linear counting with a smart weighting.  

---
```python
from sklearn.feature_extraction.text import TfidfVectorizer

vect = TfidfVectorizer(stop_words='english')
data = ['dog bites cat', 'cat bites dog', 'cat and dog house']

X = vect.fit_transform(data).toarray()
features = vect.get_feature_names_out()
```


1. TfidfVectorizer(stop_words='english')  
- Removes English stopwords (like “and”, “the”, …).
- Tokenizes the text.
- Builds a vocabulary from the remaining words.

2.	vect.fit_transform(data)  
- fit: learns the vocabulary and IDF values from data.
- transform: builds the TF–IDF matrix (documents × vocabulary).

3.	vect.get_feature_names_out()  
- With data above (and English stopwords removed), the vocabulary is:
['bites', 'cat', 'dog', 'house']

---

#### 2. PCA (Principal Component Analysis)
What is PCA?  
A method for dimensionality reduction on numeric data.  

Given data $X \in \mathbb{R}^{n \times p}$ (n samples, p features):  
- PCA:
1.	Centers the data (subtracts the mean of each feature).  
2.	Finds directions (vectors) where the variance is maximal.  
3.	These directions are the principal components (PC1, PC2, …).  
4.	You project the data onto the first k components → you get k new features that explain most of the variance.  

Mathematically:
- It finds eigenvectors of the covariance matrix.
- Sorts them by eigenvalue (variance explained).
- You keep the first k.
```python
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

pca = PCA(n_components = 5)
x_projection = pca.fit_transform(X)
plt.plot(pca.explained_variance_ratio_, marker='o', linestyle='')
# find an elbow and decide the number of components

my_classifier.train(x_projection, y_train)

x_test_proj = pca.transform(x_test)
y_test_pred = my_classifier.predict(x_test_proj)
```

Step by step:
1.	pca = PCA(n_components=5)  
You ask PCA to compute the first 5 principal components.  

2.	x_projection = pca.fit_transform(X)  
- fit: computes the components from the data X (usually X_train).  
- transform: projects X into the new 5-dimensional space --> Result: x_projection has shape (n_samples, 5).  

3.	pca.explained_variance_ratio_  
- For each component, how much of the total variance it explains.
- Example: [0.6, 0.2, 0.1, 0.05, 0.05].
- You plot this and look for an elbow (where adding more components gives diminishing returns).

4.	Train classifier on reduced data:
my_classifier.train(x_projection, y_train)  

5.	Apply the same PCA to the test set:  
x_test_proj = pca.transform(x_test)
y_test_pred = my_classifier.predict(x_test_proj)

---

1.	Scaling before PCA  
- PCA is scale sensitive
- Normal pipeline for numeric data:
```python
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

model = make_pipeline(
    StandardScaler(),
    PCA(n_components=5),
    SomeClassifier()
)
```

2. PCA doesn’t automatically “improve” the model  
- It can remove noise, reduce overfitting → sometimes better.
- It can also throw away informative dimensions → sometimes worse.
You check with cross-validation, not with faith.  

3. Choosing the number of components  
- Elbow method (from the scree plot).
- Or directly:
```python
PCA(n_components=0.95)
```
which keeps the minimum number of components that explain 95% of the variance.